# EDA: Breathing Patterns Dataset (Radiation Gating)

This notebook explores the consolidated breathing-curve data: duration, volume ranges, balloon/gating status counts per patient, and missingness.

**Note**: Run `python analysis/analyze_dataset.py` from the project root first to regenerate `analysis/output/file_summary.csv` and `patient_summary.csv` (includes .dat files).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from src.load_data import load_all_patients
import config as cfg

In [ ]:
# Load summary CSVs (generated by analyze_dataset.py)
analysis_out = Path("../analysis/output")
file_summary = pd.read_csv(analysis_out / "file_summary.csv") if (analysis_out / "file_summary.csv").exists() else None
patient_summary = pd.read_csv(analysis_out / "patient_summary.csv") if (analysis_out / "patient_summary.csv").exists() else None

if file_summary is not None:
    print("File summary shape:", file_summary.shape)
    print(file_summary["file_type"].value_counts())
if patient_summary is not None:
    print("\nPatient summary shape:", patient_summary.shape)
    print(patient_summary.head(10))

In [ ]:
# Load a sample of curve data (optional - can be large)
dataset_dir = cfg.DATASET_DIR
if dataset_dir.exists():
    df, session_meta = load_all_patients(dataset_dir, include_session_ini=False)
    print("Combined curve data shape:", df.shape)
    print("Patients:", df["patient_id"].nunique())
    print("\nVolume stats:")
    print(df["Volume (liters)"].describe())
    print("\nBalloon Valve value counts:")
    print(df["Balloon Valve Status"].astype(str).value_counts().head(10))

In [ ]:
# Duration and volume ranges per patient
if "df" in dir() and len(df) > 0:
    per_patient = df.groupby("patient_id").agg(
        rows=("Session Time", "count"),
        duration_s=("Session Time", lambda x: x.max() - x.min() if len(x) > 1 else 0),
        vol_min=("Volume (liters)", "min"),
        vol_max=("Volume (liters)", "max"),
        vol_mean=("Volume (liters)", "mean"),
    ).reset_index()
    print(per_patient.head(15))
    if "matplotlib" in sys.modules or True:
        try:
            import matplotlib.pyplot as plt
            per_patient["duration_s"].hist(bins=30)
            plt.xlabel("Duration (s)")
            plt.title("Distribution of recording duration per patient")
            plt.show()
        except Exception:
            pass

**Validation notes**: Use `patient_id` = folder name for stable train/test splits. Different folder names (e.g. "Abdul Rehaman" vs "ABDUL REHAMAN N_._7132024") are treated as different patients. Exclude very short files if needed (see `config.MIN_WINDOW_ROWS`).